# Dental Implant Survival Analysis – Andersen–Gill Cox Models

**Target journal:** Journal of Clinical Periodontology (JCP)

This notebook performs a comprehensive survival analysis of dental implants using
Andersen–Gill (AG) Cox proportional hazards models with robust standard errors
(clustered by patient). Three models are fitted:

1. **Overall AG model** – full cohort
2. **Early-failure AG model** – failures within ≤365 days
3. **Late-failure AG model** – failures after >365 days

The analysis also includes Kaplan–Meier curves with time-at-risk tables,
expanded univariable survival screening, and a global proportional-hazards
diagnostic for each AG model.

All functions are imported from `functions.py` for modularity and reproducibility.

In [ ]:
# ============================================================
# 0. Setup & Imports
# ============================================================
import importlib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project module
import functions as fn
importlib.reload(fn)

fn.print_versions()

# Display settings
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Configuration

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
INPUT_PATH = r"C:\Users\cahana_s\research\research_shetel_2_3\shetel_2_3_data\implants_with_restoration_summary_all.xlsx"
OUTDIR     = r"C:\Users\cahana_s\research\research_shetel_2_3\outputs"
CENSOR_DATE   = "2025-08-24"
COHORT_CUTOFF = "2023-07-01"
EARLY_THRESHOLD_DAYS = 365   # ≤365 = early, >365 = late

import os
os.makedirs(OUTDIR, exist_ok=True)

## 2. Data Loading & Preprocessing

In [ ]:
# ============================================================
# 2. Data Loading & Preprocessing
# ============================================================
from pathlib import Path
import pandas as pd

PARQUET_PATH = Path("final_cohort.parquet")

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
else:
    df_raw = fn.load_data(INPUT_PATH)
    df = fn.preprocess(
        df_raw,
        censor_date=CENSOR_DATE,
        cohort_cutoff=COHORT_CUTOFF
    )

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    print("Casting object columns to string:", object_cols)

    for col in object_cols:
        df[col] = df[col].astype("string").str.strip()

    df.to_parquet(PARQUET_PATH, index=False)

print(f"Final cohort: {len(df):,} implants | {df['patient_id'].nunique():,} patients")
print(f"Failures:     {int(df['is_failure'].sum()):,} ({df['is_failure'].mean()*100:.1f}%)")

## 3. Descriptive Statistics

In [ ]:
# ============================================================
# 3a. Continuous variables
# ============================================================
continuous_cols = ["age_years", "length", "diameter", "days_to_failure"]
display(fn.style_table(
    fn.continuous_summary(df, continuous_cols),
    caption="Continuous Variables"
))

In [ ]:
# ============================================================
# 3b. Categorical / binary variables
# ============================================================
cat_cols = [
    "jaw", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv_lbl",
    "rehabilitation_type", "fixed_loading_type",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    "has_mahash_onimplantday", "has_resm_onimplantday",
    "has_resp_onorbeforeimplant",
]
display(fn.style_table(
    fn.frequency_table(df, cat_cols),
    caption="Categorical and Binary Variables"
))

In [ ]:
# ============================================================
# 3c. Follow-up summary
# ============================================================
display(fn.style_table(
    fn.follow_up_summary(df),
    caption="Follow-up Summary"
))

## 4. Implant Survival by Procedure Sequence

In [ ]:
# ============================================================
# 4. Stacked bar chart – survival vs failure by implant sequence
# ============================================================
fig = fn.plot_survival_by_sequence(df, save_path=os.path.join(OUTDIR, "survival_by_sequence"))
plt.show()

## 5. Kaplan–Meier Survival Curves

Each Kaplan–Meier figure includes a time-at-risk table below the curves.
When follow-up extends beyond 10 years, the displayed x-axis is capped at 10 years.

In [ ]:
# ============================================================
# 5. KM by implant number (1, 2, 3+)
# ============================================================
df["years_to_failure"] = df["days_to_failure"] / 365.25
df["implant_group"] = df["implant_num_surv"].apply(lambda g: "3+" if g >= 3 else str(int(g)))

label_map = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

fig = fn.km_plot_by_group(
    df,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map,
    color_map=color_map,
    title="Kaplan–Meier Survival by Implant Sequence",
    save_path=os.path.join(OUTDIR, "km_by_implant_number"),
)
plt.show()

## 6. Univariable Survival Screening

In [ ]:
# ============================================================
# 6. Log-rank plus univariable Cox screening
# ============================================================
test_vars = [
    "jaw", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    # "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    # "has_mahash_onimplantday", "has_resm_onimplantday",
    # "has_resp_onorbeforeimplant",
]
lr_results = fn.logrank_all_variables(df, test_vars)
univ_results = fn.univariable_survival_summary(df, test_vars)

print("Univariable survival screening:")
display(fn.style_univariable_table(univ_results))
print(f"\nSignificant log-rank tests (p < 0.05): {int(lr_results['Significant'].sum())} / {len(lr_results)}")

## 7. Cox Model Data Preparation

In [ ]:
# ============================================================
# 7. Prepare the model frame for AG models
# ============================================================
df = fn.prepare_cox_time(df, study_end=CENSOR_DATE)
mf = fn.prepare_model_frame(df)

print(f"Model frame: {len(mf):,} rows")
print(f"Failure events: {int(mf['failure_event'].sum()):,}")

## 8. Andersen–Gill Cox Model – Overall Cohort

This model includes the **full cohort** (all implant failures regardless of timing).
Each patient may contribute multiple implants (AG framework) with robust SE via `cluster(patient_id)`.

In [ ]:
# ============================================================
# 8. AG Model – Overall
# ============================================================
dat_overall = fn.prepare_ag_data(mf)

res_overall, n_overall, ev_overall, c_overall, ph_overall = fn.run_ag_model_r(
    dat_overall, model_label="Andersen–Gill Cox Model – Overall"
)

tbl_overall = fn.result_table(
    res_overall, n_overall, ev_overall, c_overall, ph_overall,
    model_label="AG Model – Overall"
 )
diag_overall = fn.model_diagnostics_table(
    "AG Model – Overall", n_overall, ev_overall, c_overall, ph_overall
)
display(fn.style_comparison_table(diag_overall))
display(fn.style_result_table(tbl_overall))

In [ ]:
# Forest plot – Overall
fig_overall = fn.forest_plot(
    res_overall, n_overall, ev_overall, c_overall,
    model_label="Andersen–Gill Cox Model – Overall",
    save_path=os.path.join(OUTDIR, "forest_AG_overall"),
)
plt.show()

## 9. Early / Late Failure Classification

**Definition (from the original analysis):**
- **Early failure:** implant failure within ≤365 days of placement
- **Late failure:** implant failure after >365 days
- **Censored:** no failure observed during follow-up

In [ ]:
# ============================================================
# 9. Classify failures as early / late
# ============================================================
mf = fn.classify_failure_type(mf, threshold_days=EARLY_THRESHOLD_DAYS)

print("Failure type distribution:")
print(mf["failure_type"].value_counts(dropna=False))
print(f"\nEarly threshold: ≤{EARLY_THRESHOLD_DAYS} days")

## 10. Andersen–Gill Cox Model – Early Failures (≤365 days)

For this sub-model, only **early failures** (≤365 days) are treated as events.
Late failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 10. AG Model – Early Failures
# ============================================================
mf_early = mf.copy()
# Re-code: only early failures count as events
mf_early["failure_event"] = np.where(mf_early["failure_type"] == "Early", 1, 0).astype(int)

dat_early = fn.prepare_ag_data(mf_early)

res_early, n_early, ev_early, c_early, ph_early = fn.run_ag_model_r(
    dat_early, model_label="AG Cox Model – Early Failures (≤365 d)"
)

tbl_early = fn.result_table(
    res_early, n_early, ev_early, c_early, ph_early,
    model_label="AG Model – Early Failures"
 )
diag_early = fn.model_diagnostics_table(
    "AG Model – Early Failures", n_early, ev_early, c_early, ph_early
)
display(fn.style_comparison_table(diag_early))
display(fn.style_result_table(tbl_early))

In [ ]:
# Forest plot – Early
fig_early = fn.forest_plot(
    res_early, n_early, ev_early, c_early,
    model_label="AG Cox Model – Early Failures (≤365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_early"),
)
plt.show()

## 11. Andersen–Gill Cox Model – Late Failures (>365 days)

For this sub-model, only **late failures** (>365 days) are treated as events.
Early failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 11. AG Model – Late Failures
# ============================================================
mf_late = mf.copy()
# Re-code: only late failures count as events
mf_late["failure_event"] = np.where(mf_late["failure_type"] == "Late", 1, 0).astype(int)

dat_late = fn.prepare_ag_data(mf_late)

res_late, n_late, ev_late, c_late, ph_late = fn.run_ag_model_r(
    dat_late, model_label="AG Cox Model – Late Failures (>365 d)"
)

tbl_late = fn.result_table(
    res_late, n_late, ev_late, c_late, ph_late,
    model_label="AG Model – Late Failures"
 )
diag_late = fn.model_diagnostics_table(
    "AG Model – Late Failures", n_late, ev_late, c_late, ph_late
)
display(fn.style_comparison_table(diag_late))
display(fn.style_result_table(tbl_late))

In [ ]:
# Forest plot – Late
fig_late = fn.forest_plot(
    res_late, n_late, ev_late, c_late,
    model_label="AG Cox Model – Late Failures (>365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_late"),
)
plt.show()

## 12. Model Comparison Summary and Diagnostics

In [ ]:
# ============================================================
# 12. Side-by-side model comparison
# ============================================================
comparison = pd.concat([diag_overall, diag_early, diag_late], ignore_index=True)
display(fn.style_comparison_table(comparison))

# Merge HR tables side-by-side for the three models
merged = (
    res_overall[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Overall)", "p_fmt": "p (Overall)"})
    .merge(
        res_early[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Early)", "p_fmt": "p (Early)"}),
        on="label", how="outer"
    )
    .merge(
        res_late[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Late)", "p_fmt": "p (Late)"}),
        on="label", how="outer"
    )
)
merged = merged.rename(columns={"label": "Variable"})
display(fn.style_comparison_table(merged))

## 13. Export Results

In [ ]:
# ============================================================
# 13. Save tables to Excel
# ============================================================
excel_path = os.path.join(OUTDIR, "AG_model_results.xlsx")

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    tbl_overall.to_excel(writer, sheet_name="Overall", index=False)
    tbl_early.to_excel(writer, sheet_name="Early", index=False)
    tbl_late.to_excel(writer, sheet_name="Late", index=False)
    merged.to_excel(writer, sheet_name="Comparison", index=False)
    comparison.to_excel(writer, sheet_name="Summary", index=False)
    univ_results.to_excel(writer, sheet_name="Univariable", index=False)
    lr_results.to_excel(writer, sheet_name="LogRank", index=False)

print(f"Results saved to: {excel_path}")
print(f"Forest plots saved to: {OUTDIR}")
print("\nDone. All outputs ready for JCP submission.")